# 01 Frame Extraction

This notebook extracts or reviews residence video frames.

It writes only the raw frame folder, frame manifests, and frame contact sheet. It does not create or update any grounding benchmark CSV.

Main outputs:

- `examples/residence_images/`
- `examples/image_manifest_extracted.csv`
- `results/contact_sheets/frame_contact_sheet.jpg`

The shared project package may not contain MP4 files. Leave `RUN_FRAME_EXTRACTION = False` to rebuild manifests and the contact sheet from the existing residence images. Set it to `True` only when the source videos are restored.


In [ ]:
# Colab setup.
# Pillow is intentionally not reinstalled here. Reinstalling Pillow after it has
# already been imported can create a mixed PIL installation in the active runtime.

%pip -q install --upgrade-strategy only-if-needed pandas opencv-python-headless

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Drive is already mounted or this is not a Colab runtime.')

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import Image as IPImage, display
import cv2
import math
import numpy as np
import pandas as pd
import shutil
import subprocess
import tempfile

print('Pillow version:', Image.__version__)


In [ ]:
# Project location and extraction settings.

PROJECT_ROOT = Path('/content/drive/MyDrive/autonomous-delivery-robot/modules/perception/visual_grounding')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}. Update PROJECT_ROOT above.')

# The shared zip does not require the MP4 files. Change to True only when videos exist.
RUN_FRAME_EXTRACTION = False
OVERWRITE_EXISTING_FRAMES = False

VIDEO_MANIFEST_PATH = PROJECT_ROOT / 'examples' / 'video_manifest.csv'
FRAME_PLAN_PATH = PROJECT_ROOT / 'examples' / 'frame_extraction_plan.csv'
VIDEO_DIR = PROJECT_ROOT / 'examples' / 'residence_videos'
FRAME_DIR = PROJECT_ROOT / 'examples' / 'residence_images'
IMAGE_MANIFEST_EXTRACTED_PATH = PROJECT_ROOT / 'examples' / 'image_manifest_extracted.csv'
CONTACT_SHEET_DIR = PROJECT_ROOT / 'results' / 'contact_sheets'

IMAGE_FORMAT = 'jpg'
JPEG_QUALITY = 98

USE_SHARPEST_FRAME_IN_WINDOW = True
FRAME_SEARCH_WINDOW_SECONDS = 0.6
FRAME_SEARCH_STEP_SECONDS = 0.1

FRAME_COUNT_MODE = 'fixed'
FRAMES_PER_VIDEO = 3

CONTACT_SHEET_THUMB_WIDTH = 480
CONTACT_SHEET_COLUMNS = 3

FRAME_DIR.mkdir(parents=True, exist_ok=True)
CONTACT_SHEET_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Frame extraction enabled:', RUN_FRAME_EXTRACTION)


In [ ]:
def relpath(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace('\\', '/')
    except Exception:
        return str(path).replace('\\', '/')


def resolve_project_path(path_value):
    if pd.isna(path_value):
        return None

    p = Path(str(path_value))
    if p.is_absolute() and p.exists():
        return p

    candidate = PROJECT_ROOT / p
    if candidate.exists():
        return candidate

    candidate = VIDEO_DIR / p.name
    if candidate.exists():
        return candidate

    return PROJECT_ROOT / p


def command_exists(command):
    return shutil.which(command) is not None


def run_command(command):
    return subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )


def ffprobe_duration(video_path):
    if not command_exists('ffprobe'):
        return None

    cmd = [
        'ffprobe',
        '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(video_path),
    ]
    result = run_command(cmd)

    if result.returncode != 0:
        return None

    try:
        return float(result.stdout.strip())
    except Exception:
        return None


def ffprobe_fps(video_path):
    if not command_exists('ffprobe'):
        return None

    cmd = [
        'ffprobe',
        '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=avg_frame_rate',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(video_path),
    ]
    result = run_command(cmd)

    if result.returncode != 0:
        return None

    value = result.stdout.strip()

    try:
        if '/' in value:
            a, b = value.split('/')
            return float(a) / max(float(b), 1.0)
        return float(value)
    except Exception:
        return None


def cv2_duration_and_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    if fps and fps > 0 and frame_count and frame_count > 0:
        return frame_count / fps, fps

    return None, fps if fps and fps > 0 else None


def get_video_info(video_path):
    duration = ffprobe_duration(video_path)
    fps = ffprobe_fps(video_path)

    if duration is None or fps is None:
        cv_duration, cv_fps = cv2_duration_and_fps(video_path)
        duration = duration if duration is not None else cv_duration
        fps = fps if fps is not None else cv_fps

    return duration, fps


def extract_frame_ffmpeg(video_path, timestamp_sec):
    if not command_exists('ffmpeg'):
        return None

    timestamp_sec = max(0.0, float(timestamp_sec))

    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
        tmp_path = Path(tmp.name)

    cmd = [
        'ffmpeg',
        '-hide_banner',
        '-loglevel', 'error',
        '-ss', f'{timestamp_sec:.3f}',
        '-i', str(video_path),
        '-frames:v', '1',
        '-f', 'image2',
        '-vcodec', 'png',
        str(tmp_path),
        '-y',
    ]

    result = run_command(cmd)

    if result.returncode != 0 or not tmp_path.exists() or tmp_path.stat().st_size == 0:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass
        return None

    try:
        image = Image.open(tmp_path).convert('RGB')
    finally:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass

    return image


def extract_frame_cv2(video_path, timestamp_sec):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, float(timestamp_sec)) * 1000.0)
    ok, frame_bgr = cap.read()
    cap.release()

    if not ok or frame_bgr is None:
        return None

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(frame_rgb)


def extract_frame(video_path, timestamp_sec):
    image = extract_frame_ffmpeg(video_path, timestamp_sec)
    if image is not None:
        return image, 'ffmpeg'

    image = extract_frame_cv2(video_path, timestamp_sec)
    if image is not None:
        return image, 'opencv'

    return None, 'failed'


def sharpness_score(image):
    arr = np.array(image.convert('RGB'))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def brightness_mean(image):
    arr = np.array(image.convert('RGB'))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    return float(gray.mean())


def candidate_times(target_time, duration):
    if not USE_SHARPEST_FRAME_IN_WINDOW:
        return [max(0.0, min(float(target_time), float(duration or target_time)))]

    half_window = FRAME_SEARCH_WINDOW_SECONDS / 2.0
    count = int(round(FRAME_SEARCH_WINDOW_SECONDS / FRAME_SEARCH_STEP_SECONDS)) + 1
    times = []

    for i in range(count):
        t = float(target_time) - half_window + i * FRAME_SEARCH_STEP_SECONDS
        if duration is not None:
            t = max(0.0, min(t, max(0.0, float(duration) - 0.001)))
        else:
            t = max(0.0, t)
        times.append(round(t, 3))

    return sorted(set(times))


def select_frame_near_time(video_path, target_time, duration):
    best = None

    for t in candidate_times(target_time, duration):
        image, backend = extract_frame(video_path, t)
        if image is None:
            continue

        brightness = brightness_mean(image)
        if brightness < 8 or brightness > 247:
            continue

        score = sharpness_score(image)

        if best is None or score > best['sharpness_score']:
            best = {
                'image': image,
                'selected_time_sec': float(t),
                'target_time_sec': float(target_time),
                'sharpness_score': score,
                'backend': backend,
                'brightness_mean': brightness,
            }

    if best is None:
        image, backend = extract_frame(video_path, target_time)
        if image is None:
            return None
        best = {
            'image': image,
            'selected_time_sec': float(target_time),
            'target_time_sec': float(target_time),
            'sharpness_score': sharpness_score(image),
            'backend': backend,
            'brightness_mean': brightness_mean(image),
        }

    return best


def save_frame_image(image, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if IMAGE_FORMAT.lower() == 'png':
        image.save(output_path)
        return

    image.save(
        output_path,
        quality=JPEG_QUALITY,
        subsampling=0,
        optimize=True,
    )


def get_frame_count(video_id, frame_plan_df=None):
    if FRAME_COUNT_MODE == 'frame_plan' and frame_plan_df is not None and 'video_id' in frame_plan_df.columns:
        matches = frame_plan_df[frame_plan_df['video_id'].astype(str) == str(video_id)]
        if len(matches) and 'frames_to_extract' in matches.columns:
            value = matches.iloc[0]['frames_to_extract']
            try:
                return max(1, int(value))
            except Exception:
                pass

    return int(FRAMES_PER_VIDEO)


def planned_times(duration, frame_count):
    if duration is None or duration <= 0:
        return [float(i) for i in range(frame_count)]

    if frame_count == 1:
        return [duration / 2.0]

    return [duration * (i + 1) / (frame_count + 1) for i in range(frame_count)]


def make_image_id(video_row, frame_number):
    video_path_value = str(video_row.get('video_path', ''))
    stem = Path(video_path_value).stem

    if not stem or stem == '.':
        video_id = str(video_row.get('video_id', 'video')).strip()
        scenario = str(video_row.get('scenario_label', '')).strip()
        stem = video_id if not scenario else f'{video_id}_{scenario}'

    return f'{stem}_frame_{frame_number:02d}'


In [ ]:
def load_video_manifest():
    if VIDEO_MANIFEST_PATH.exists():
        df = pd.read_csv(VIDEO_MANIFEST_PATH)
    else:
        videos = sorted(VIDEO_DIR.glob('*.mp4'))
        df = pd.DataFrame({
            'video_id': [p.stem.split('_')[0] if '_' in p.stem else p.stem for p in videos],
            'video_path': [relpath(p) for p in videos],
            'scenario_label': [p.stem for p in videos],
            'source': 'residence',
        })

    if 'video_path' not in df.columns:
        raise ValueError('video_manifest.csv must contain a video_path column.')

    return df


def load_frame_plan():
    if FRAME_PLAN_PATH.exists():
        return pd.read_csv(FRAME_PLAN_PATH)
    return None


def extract_frames_from_manifest():
    video_manifest = load_video_manifest()
    frame_plan = load_frame_plan()

    rows = []
    skipped_videos = []

    for _, video_row in video_manifest.iterrows():
        video_path = resolve_project_path(video_row['video_path'])
        video_id = str(video_row.get('video_id', Path(str(video_row['video_path'])).stem))

        if video_path is None or not video_path.exists():
            skipped_videos.append({'video_id': video_id, 'video_path': str(video_path), 'reason': 'video file not found'})
            continue

        duration, fps = get_video_info(video_path)
        frame_count = get_frame_count(video_id, frame_plan)
        times = planned_times(duration, frame_count)

        for frame_number, target_time in enumerate(times, start=1):
            image_id = make_image_id(video_row, frame_number)
            output_path = FRAME_DIR / f'{image_id}.{IMAGE_FORMAT}'

            selected = None

            if output_path.exists() and not OVERWRITE_EXISTING_FRAMES:
                image = Image.open(output_path).convert('RGB')
                selected = {
                    'image': image,
                    'selected_time_sec': float(target_time),
                    'target_time_sec': float(target_time),
                    'sharpness_score': None,
                    'backend': 'existing_file',
                    'brightness_mean': None,
                }
            else:
                selected = select_frame_near_time(video_path, target_time, duration)
                if selected is None:
                    rows.append({
                        'image_id': image_id,
                        'source': 'residence_video_frame',
                        'image_path': relpath(output_path),
                        'video_id': video_id,
                        'video_path': relpath(video_path),
                        'frame_index': '',
                        'timestamp_sec': '',
                        'target_timestamp_sec': round(float(target_time), 3),
                        'selected_timestamp_sec': '',
                        'extraction_backend': 'failed',
                        'extraction_note': 'frame extraction failed',
                    })
                    continue

                save_frame_image(selected['image'], output_path)

            frame_index = ''
            if fps is not None and selected['selected_time_sec'] is not None:
                frame_index = int(round(selected['selected_time_sec'] * fps))

            row = {
                'image_id': image_id,
                'source': 'residence_video_frame',
                'image_path': relpath(output_path),
                'video_id': video_id,
                'video_path': relpath(video_path),
                'scene_class': video_row.get('scene_class', ''),
                'target_type': video_row.get('target_type', ''),
                'target_object': video_row.get('target_object', ''),
                'command_id': f'cmd_{video_id.replace("vid_", "")}' if str(video_id).startswith('vid_') else f'cmd_{video_id}',
                'grounding_prompt': video_row.get('grounding_prompt', ''),
                'frame_index': frame_index,
                'timestamp_sec': round(float(selected['selected_time_sec']), 3),
                'target_timestamp_sec': round(float(selected['target_time_sec']), 3),
                'selected_timestamp_sec': round(float(selected['selected_time_sec']), 3),
                'extraction_backend': selected['backend'],
                'privacy_checked': video_row.get('privacy_checked', 'yes'),
                'notes': video_row.get('notes', ''),
                'extraction_note': 'sharpest nearby frame selected' if USE_SHARPEST_FRAME_IN_WINDOW else 'planned timestamp frame selected',
            }
            rows.append(row)

    extracted_df = pd.DataFrame(rows)

    if len(skipped_videos):
        print('Videos skipped because files were not found:')
        display(pd.DataFrame(skipped_videos))

    return extracted_df


def load_existing_manifest():
    """Build a current manifest from files that actually exist in residence_images.

    Existing manifest metadata is reused only for matching image IDs. Deleted or
    renamed frame files cannot remain as stale manifest rows.
    """
    images = sorted([
        p for p in FRAME_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']
    ])

    if not images:
        return pd.DataFrame(columns=[
            'image_id', 'source', 'image_path', 'video_id', 'video_path',
            'frame_index', 'timestamp_sec', 'privacy_checked', 'notes',
        ])

    existing = None
    for path in [IMAGE_MANIFEST_EXTRACTED_PATH]:
        if path.exists():
            candidate = pd.read_csv(path)
            if 'image_id' in candidate.columns:
                existing = candidate.drop_duplicates('image_id', keep='last').set_index('image_id')
                break

    rows = []
    for p in images:
        image_id = p.stem
        row = {
            'image_id': image_id,
            'source': 'residence_video_frame',
            'image_path': relpath(p),
            'video_id': image_id.split('_frame_')[0] if '_frame_' in image_id else '',
            'video_path': '',
            'frame_index': '',
            'timestamp_sec': '',
            'privacy_checked': 'yes',
            'notes': '',
        }

        if existing is not None and image_id in existing.index:
            old = existing.loc[image_id]
            if isinstance(old, pd.DataFrame):
                old = old.iloc[-1]
            for column, value in old.to_dict().items():
                if column not in {'image_id', 'image_path'} and pd.notna(value):
                    row[column] = value

        # The current file system is always authoritative for these fields.
        row['image_id'] = image_id
        row['image_path'] = relpath(p)
        rows.append(row)

    return pd.DataFrame(rows)


def save_manifests(image_manifest):
    image_manifest = image_manifest.copy()
    image_manifest = image_manifest.drop_duplicates(subset=['image_id'], keep='last')
    image_manifest = image_manifest.sort_values('image_id').reset_index(drop=True)

    output_paths = [IMAGE_MANIFEST_EXTRACTED_PATH]

    for path in output_paths:
        path.parent.mkdir(parents=True, exist_ok=True)
        image_manifest.to_csv(path, index=False)

    return image_manifest



def make_contact_sheet(image_manifest, output_path):
    image_paths = []

    for _, row in image_manifest.iterrows():
        p = PROJECT_ROOT / str(row['image_path'])
        if p.exists() and p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
            image_paths.append(p)

    if not image_paths:
        print('No images found for contact sheet.')
        return None

    thumbs = []
    resample = getattr(Image, 'Resampling', Image).LANCZOS

    for p in image_paths:
        img = Image.open(p).convert('RGB')
        w, h = img.size
        new_h = int(h * (CONTACT_SHEET_THUMB_WIDTH / max(1, w)))
        thumb = img.resize((CONTACT_SHEET_THUMB_WIDTH, new_h), resample)
        thumbs.append((p.name, thumb))

    cols = CONTACT_SHEET_COLUMNS
    pad = 14
    label_h = 30
    rows = math.ceil(len(thumbs) / cols)
    max_h = max(t.height for _, t in thumbs)

    sheet_w = cols * CONTACT_SHEET_THUMB_WIDTH + (cols + 1) * pad
    sheet_h = rows * (max_h + label_h + pad) + pad

    sheet = Image.new('RGB', (sheet_w, sheet_h), 'white')
    draw = ImageDraw.Draw(sheet)

    for idx, (name, thumb) in enumerate(thumbs):
        r = idx // cols
        c = idx % cols
        x = pad + c * (CONTACT_SHEET_THUMB_WIDTH + pad)
        y = pad + r * (max_h + label_h + pad)
        sheet.paste(thumb, (x, y))
        draw.text((x, y + thumb.height + 5), name[:58], fill=(0, 0, 0))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, quality=95, subsampling=0, optimize=True)
    return output_path


In [ ]:
if RUN_FRAME_EXTRACTION:
    extracted_manifest = extract_frames_from_manifest()

    if len(extracted_manifest) == 0:
        print('No new frames were extracted. Existing manifest will be used if available.')
        image_manifest = load_existing_manifest()
    else:
        image_manifest = extracted_manifest
else:
    image_manifest = load_existing_manifest()

image_manifest = save_manifests(image_manifest)

contact_sheet_path = CONTACT_SHEET_DIR / 'frame_contact_sheet.jpg'
make_contact_sheet(image_manifest, contact_sheet_path)

print(f'Frame manifest rows: {len(image_manifest)}')
print(f'Frame folder: {FRAME_DIR}')
print(f'Contact sheet: {contact_sheet_path}')


preview_cols = [c for c in [
    'image_id',
    'image_path',
    'video_id',
    'timestamp_sec',
    'target_timestamp_sec',
    'selected_timestamp_sec',
    'extraction_backend',
    'grounding_prompt',
] if c in image_manifest.columns]

display(image_manifest[preview_cols].head(12))


# Notebook 01 output contract.
required_manifest_columns = {'image_id', 'image_path'}
missing_manifest_columns = required_manifest_columns - set(image_manifest.columns)
if missing_manifest_columns:
    raise ValueError(f'Generated manifest is missing columns: {sorted(missing_manifest_columns)}')

missing_frame_paths = []
for image_path_value in image_manifest['image_path'].astype(str):
    path = Path(image_path_value)
    resolved = path if path.is_absolute() else PROJECT_ROOT / path
    if not resolved.exists():
        missing_frame_paths.append(str(resolved))

if missing_frame_paths:
    raise FileNotFoundError(
        'Manifest contains frame paths that do not exist:\n' + '\n'.join(missing_frame_paths[:20])
    )

for required_output in [IMAGE_MANIFEST_EXTRACTED_PATH]:
    if not required_output.exists():
        raise FileNotFoundError(f'Notebook 01 did not create: {required_output}')

print('Notebook 01 integration contract passed.')
print('Notebook 02 will read raw frames from:', FRAME_DIR)


In [ ]:
contact_sheet_path = CONTACT_SHEET_DIR / 'frame_contact_sheet.jpg'

if contact_sheet_path.exists():
    display(IPImage(filename=str(contact_sheet_path)))
else:
    print('Contact sheet was not created.')
